In [1]:
import sys

import matplotlib
import numpy
import pandas
import sklearn

print(sys.version.split()[0], numpy.__version__, pandas.__version__, sklearn.__version__, matplotlib.__version__)

3.12.6 2.3.4 2.3.3 1.7.2 3.10.7


In [2]:
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold, train_test_split

SEED = 3012          
data = pd.read_csv("data/spambase.csv")
feature_names = [c for c in data.columns if c != "spam"]
X, y = data[feature_names].to_numpy(), data["spam"].to_numpy()
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=SEED)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
print(X.shape, round(100 * y.mean(), 1), X_train.shape, X_test.shape)

(4204, 57) 39.9 (3153, 57) (1051, 57)


In [3]:
import sys, os
print(sys.executable)

c:\Users\User\Downloads\task-04-ML-course-main (1)\zhumanov_ML_task_4\.zhumanov_ML_task_4\Scripts\python.exe


SEED = 3012

In [ ]:
from sklearn.ensemble import AdaBoostClassifier, BaggingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler
from sklearn.tree import DecisionTreeClassifier

tree = DecisionTreeClassifier(random_state=SEED)
bag = BaggingClassifier(DecisionTreeClassifier(), n_estimators=100, oob_score=True, random_state=SEED)
rf = RandomForestClassifier(n_estimators=300, oob_score=True, random_state=SEED, n_jobs=-1)
lr = make_pipeline(FunctionTransformer(np.log1p), StandardScaler(), LogisticRegression(max_iter=2000))
bag_lr = BaggingClassifier(lr, n_estimators=50, random_state=SEED)
knn = make_pipeline(FunctionTransformer(np.log1p), StandardScaler(), KNeighborsClassifier(n_neighbors=15))
ada = AdaBoostClassifier(n_estimators=400, random_state=SEED)
members = [("lr", lr), ("knn", knn), ("tree", tree)]  

# Part 1

In [5]:
import numpy as np

def oob_mask(sample_indices, n_samples):
    """Boolean mask of the rows that do NOT occur in sample_indices."""
    mask = np.ones(n_samples, dtype=bool)
    idx = np.asarray(sample_indices, dtype=int)
    mask[idx] = False
    return mask

In [6]:
print(oob_mask([0, 2, 2], 5))
print(oob_mask([], 3))
print(oob_mask(np.array([3, 0]), 4))

[False  True False  True  True]
[ True  True  True]
[False  True  True False]


In [7]:
bag.fit(X_train, y_train)
print(len(bag.estimators_), len(bag.estimators_samples_))

100 100


In [8]:
n = len(X_train)

oob_shares = [
    oob_mask(idx, n).mean()
    for idx in bag.estimators_samples_
]
mean_oob_share = np.mean(oob_shares)
mean_in_share = 1 - mean_oob_share

formula_oob = (1 - 1 / n) ** n
formula_in = 1 - formula_oob

print("mean out-of-bag share:", mean_oob_share)
print("mean in-bag share:    ", mean_in_share)
print("formula (1-1/n)^n:    ", formula_oob)
print("formula 1-(1-1/n)^n:  ", formula_in)

mean out-of-bag share: 0.36847446875991124
mean in-bag share:     0.6315255312400887
formula (1-1/n)^n:     0.3678210954587638
formula 1-(1-1/n)^n:   0.6321789045412363


In [9]:
n = len(X_train)
votes = np.zeros((n, 2), dtype=int)

for tree_k, idx in zip(bag.estimators_, bag.estimators_samples_):
    mask = oob_mask(idx, n)
    if mask.any():
        pred = tree_k.predict(X_train[mask])
        rows = np.where(mask)[0]
        votes[rows, pred.astype(int)] += 1

oob_pred = np.argmax(votes, axis=1)   # tie -> class 0
my_oob_acc = (oob_pred == y_train).mean()

print("my OOB accuracy:  ", my_oob_acc)
print("bag.oob_score_:   ", bag.oob_score_)
print("rows with no OOB votes:", (votes.sum(axis=1) == 0).sum())

my OOB accuracy:   0.937836980653346
bag.oob_score_:    0.937836980653346
rows with no OOB votes: 0


In [10]:
from sklearn.model_selection import cross_validate

models = {"tree": tree, "bag": bag, "rf": rf, "lr": lr, "bag_lr": bag_lr}
fold_scores = {}
for name, model in models.items():
    fold_scores[name] = cross_validate(
        model, X_train, y_train, cv=cv, scoring="accuracy"
    )["test_score"]

table = pd.DataFrame(fold_scores, index=[f"fold {i+1}" for i in range(5)]).T
table["mean"] = [s.mean() for s in fold_scores.values()]
table["std"] = [np.std(s) for s in fold_scores.values()]
print(table.round(4))

        fold 1  fold 2  fold 3  fold 4  fold 5    mean     std
tree    0.8970  0.9160  0.8811  0.8778  0.9079  0.8960  0.0148
bag     0.9334  0.9366  0.9303  0.9381  0.9413  0.9359  0.0038
rf      0.9414  0.9461  0.9414  0.9460  0.9571  0.9464  0.0058
lr      0.9350  0.9461  0.9366  0.9286  0.9333  0.9359  0.0058
bag_lr  0.9414  0.9429  0.9319  0.9286  0.9349  0.9359  0.0055


In [11]:
def paired(a, b):
    d = fold_scores[a] - fold_scores[b]
    m, s = d.mean(), np.std(d)
    print(f"{a} - {b}: folds={np.round(d, 4)}  mean={m:.4f}  std={s:.4f}  "
          f"inside spread: {abs(m) <= s}")

paired("bag", "tree")
paired("rf", "tree")
paired("bag_lr", "lr")

bag - tree: folds=[0.0365 0.0206 0.0491 0.0603 0.0333]  mean=0.0400  std=0.0136  inside spread: False
rf - tree: folds=[0.0444 0.0301 0.0602 0.0683 0.0492]  mean=0.0504  std=0.0131  inside spread: False
bag_lr - lr: folds=[ 0.0063 -0.0032 -0.0048  0.      0.0016]  mean=0.0000  std=0.0039  inside spread: True


**(a) Does bagging change cross-validation accuracy?**

For the decision tree, the paired fold differences bag − tree are [0.0365, 0.0206, 0.0491, 0.0603, 0.0333], with mean 0.0400 and std 0.0136. The absolute mean is larger than the std, so the difference is outside the fold spread: bagging improves the tree by about 4 percentage points, and all five folds agree in sign. A single fully grown tree has high variance: it fits the noise of its training sample and changes a lot when the sample changes. Averaging 100 trees trained on different bootstrap samples cancels much of this noise, while the bias stays almost the same.

For logistic regression, the paired differences bag_lr − lr are [0.0063, -0.0032, -0.0048, 0.0000, 0.0016], with mean 0.0000 and std 0.0039. The absolute mean is not larger than the std, so the difference is inside the fold spread, and the signs differ between folds. My data cannot separate bagged logistic regression from plain logistic regression. Logistic regression is a stable, low-variance learner, so there is little variance to average away. Each bootstrap copy also sees only about 63% of the distinct rows, which makes each member slightly weaker, and this roughly cancels the small gain from averaging.

For comparison, rf − tree has mean 0.0504 and std 0.0131, so the forest is also clearly better than the single tree. Random feature selection at each split makes the trees less correlated than in plain bagging, which is why the forest does even better.

In [12]:
import math

rf.fit(X_train, y_train)

oob_rf = rf.oob_score_
cv_mean = fold_scores["rf"].mean()
cv_std = np.std(fold_scores["rf"])

p = rf.score(X_test, y_test)
n_test = len(y_test)
se = math.sqrt(p * (1 - p) / n_test)

print("oob_score_:       ", round(oob_rf, 4))
print("CV mean ± std:    ", round(cv_mean, 4), "±", round(cv_std, 4))
print("test accuracy p:  ", round(p, 4), " SE:", round(se, 4))
print("|oob - cv mean|:  ", round(abs(oob_rf - cv_mean), 4))
print("agree (<= CV std):", abs(oob_rf - cv_mean) <= cv_std)

oob_score_:        0.9486
CV mean ± std:     0.9464 ± 0.0058
test accuracy p:   0.9496  SE: 0.0067
|oob - cv mean|:   0.0022
agree (<= CV std): True


**(b) Is the forest's out-of-bag score a fair stand-in for its cross-validation score?**

The forest's oob_score_ is 0.9486 and its five-fold cross-validation mean is 0.9464 (std 0.0058 over the folds). The absolute difference is 0.0022, which is not larger than the fold std of 0.0058, so by the rule of this task the two numbers agree. Here the out-of-bag score is a fair stand-in for the cross-validation score: it comes from a single fit, uses each training row only through trees that never saw it, and so costs about one fit instead of five.

On the test split the forest has accuracy p = 0.9496 with standard error sqrt(p(1 - p)/n_test) = 0.0067. The test accuracy differs from the out-of-bag score by 0.0010 and from the cross-validation mean by 0.0032, both well inside one standard error. So all three estimates (0.9486, 0.9464, 0.9496) are statistically indistinguishable on this data, and none of them suggests that the forest overfits the training split.